# 🧠 Detección de Enfermedades Retinianas con Deep Learning
### ML Pipeline completo

**Problema:** Clasificación de retinopatía diabética en imágenes del fondo de ojo  
**Dataset:** APTOS 2019 Blindness Detection (Kaggle)  
**Objetivo:** Clasificar la severidad de retinopatía en 5 niveles (0=No DR, 4=Proliferative DR)

---
### 📋 Estado del Pipeline

| Etapa | Estado |
|---|---|
| ✅ 1. Data Retrieval | Completo |
| ✅ 2. Data Processing & Wrangling | Completo |
| ✅ 3. Feature Extraction & Engineering | Completo |
| ✅ 4. Feature Scaling & Selection | Completo |
| ✅ 5. Modeling (ML Algorithm) | Completo |
| ✅ 6. Model Evaluation & Tuning | Completo |
| ✅ 7. Deployment & Monitoring | Completo |

---

## 📦 0. Instalación de Dependencias

In [1]:
# Instalar librerías necesarias
!pip install -q kaggle efficientnet opencv-python-headless
!pip install -q imbalanced-learn seaborn scikit-learn matplotlib pandas numpy

ERROR: Could not find a version that satisfies the requirement tensorflow-addons (from versions: none)
ERROR: No matching distribution found for tensorflow-addons


In [2]:
# Importaciones generales
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import warnings
warnings.filterwarnings('ignore')

# Deep Learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB4
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Machine Learning clásico
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import SMOTE

# Verificar GPU
print('TensorFlow version:', tf.__version__)
print('GPU disponible:', tf.config.list_physical_devices('GPU'))
print('Dispositivo activo:', tf.test.gpu_device_name() or 'CPU')

TensorFlow version: 2.19.0
GPU disponible: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Dispositivo activo: /device:GPU:0


---
## 🗄️ 1. DATA RETRIEVAL
> **Objetivo:** Obtener el dataset APTOS 2019 desde Kaggle y verificar su integridad.

In [14]:
# ──────────────────────────────────────────────
# 1.1 Configuración de Kaggle API
# ──────────────────────────────────────────────
# Archivo kaggle.json (obtenido de https://www.kaggle.com/account)
from google.colab import files

print('Archivo kaggle.json:')
uploaded = files.upload()

# Configurar credenciales
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print('✅ Kaggle configurado correctamente')

Archivo kaggle.json:


Saving kaggle.json to kaggle (2).json
✅ Kaggle configurado correctamente


In [19]:
# Alternativa — dataset APTOS en formato ya procesado (más liviano)
!kaggle datasets download -d sovitrath/diabetic-retinopathy-224x224-2019-data -p /content/data --unzip
print('✅ Dataset descargado y descomprimido')

Dataset URL: https://www.kaggle.com/datasets/sovitrath/diabetic-retinopathy-224x224-2019-data
License(s): CC0-1.0
 96% 228M/238M [00:00<00:00, 639MB/s] 
100% 238M/238M [00:00<00:00, 654MB/s]
✅ Dataset descargado y descomprimido


In [21]:
import os
for root, dirs, files in os.walk('/content/data'):
    level = root.replace('/content/data', '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 3:
        for file in list(files)[:3]:
            print(f'{indent}  {file}')

data/
  train.csv
  colored_images/
    Proliferate_DR/
      ba735b286d62.png
      e3ec668f6fad.png
      034cb07a550f.png
    Severe/
      6735931000ec.png
      237c078d00fc.png
      e1900014dabf.png
    No_DR/
      9e2ba2b979f1.png
      c704bd669f36.png
      dce73d90c00c.png
    Moderate/
      2fdffb6160a6.png
      aad0c0ee9268.png
      a56230242a95.png
    Mild/
      ee78ce914066.png
      578109578b46.png
      3dbc90c7ee7d.png


In [22]:
# ──────────────────────────────────────────────
# 1.3 Carga y verificación inicial (estructura sovitrath)
# ──────────────────────────────────────────────
import os
import pandas as pd

BASE_DIR      = '/content/data'
IMG_DIR       = os.path.join(BASE_DIR, 'colored_images')

# Mapeo de carpetas a etiquetas numéricas
FOLDER_MAP = {
    'No_DR'         : 0,
    'Mild'          : 1,
    'Moderate'      : 2,
    'Severe'        : 3,
    'Proliferate_DR': 4
}

LABEL_MAP = {
    0: 'No DR',
    1: 'Mild',
    2: 'Moderate',
    3: 'Severe',
    4: 'Proliferative DR'
}

# Construir DataFrame desde las carpetas
rows = []
for folder, label in FOLDER_MAP.items():
    folder_path = os.path.join(IMG_DIR, folder)
    for fname in os.listdir(folder_path):
        if fname.endswith('.png'):
            rows.append({
                'id_code'   : fname.replace('.png', ''),
                'diagnosis' : label,
                'image_path': os.path.join(folder_path, fname)
            })

df_train = pd.DataFrame(rows)
df_train['diagnosis_label'] = df_train['diagnosis'].map(LABEL_MAP)

print('=' * 50)
print(f'Total imágenes encontradas: {len(df_train)}')
print('=' * 50)
print('\nDistribución por clase:')
print(df_train['diagnosis_label'].value_counts())
display(df_train.head(10))
print('\n✅ Dataset cargado correctamente')

Total imágenes encontradas: 3662

Distribución por clase:
diagnosis_label
No DR               1805
Moderate             999
Mild                 370
Proliferative DR     295
Severe               193
Name: count, dtype: int64


,id_code,diagnosis,image_path,diagnosis_label
0,9e2ba2b979f1,0,/content/data/colored_images/No_DR/9e2ba2b979f...,No DR
1,c704bd669f36,0,/content/data/colored_images/No_DR/c704bd669f3...,No DR
2,dce73d90c00c,0,/content/data/colored_images/No_DR/dce73d90c00...,No DR
3,005b95c28852,0,/content/data/colored_images/No_DR/005b95c2885...,No DR
4,9858cc2ae073,0,/content/data/colored_images/No_DR/9858cc2ae07...,No DR
5,9d62478042b6,0,/content/data/colored_images/No_DR/9d62478042b...,No DR
6,ecad6845f630,0,/content/data/colored_images/No_DR/ecad6845f63...,No DR
7,e907d23cce3d,0,/content/data/colored_images/No_DR/e907d23cce3...,No DR
8,33b91def2035,0,/content/data/colored_images/No_DR/33b91def203...,No DR
9,38373431d996,0,/content/data/colored_images/No_DR/38373431d99...,No DR



✅ Dataset cargado correctamente


---
## 🧹 2. DATA PROCESSING & WRANGLING
> **Objetivo:** Limpiar los datos, analizar distribuciones, tratar imbalance de clases y preparar splits.

In [ ]:
# ──────────────────────────────────────────────
# 2.1 Análisis Exploratorio de Datos (EDA)
# ──────────────────────────────────────────────
print('--- Información general ---')
print(df_train.info())
print('\n--- Valores nulos ---')
print(df_train.isnull().sum())
print('\n--- Estadísticas descriptivas ---')
display(df_train.describe())

In [ ]:
# ──────────────────────────────────────────────
# 2.2 Visualización de distribución de clases
# ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Conteo por clase
class_counts = df_train['diagnosis_label'].value_counts()
colors = ['#2ecc71', '#f39c12', '#e67e22', '#e74c3c', '#8e44ad']

axes[0].bar(class_counts.index, class_counts.values, color=colors)
axes[0].set_title('Distribución de Clases (Retinopatía Diabética)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Nivel de Severidad')
axes[0].set_ylabel('Cantidad de imágenes')
axes[0].tick_params(axis='x', rotation=15)
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 20, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(class_counts.values, labels=class_counts.index,
            autopct='%1.1f%%', colors=colors, startangle=90)
axes[1].set_title('Proporción de Clases', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('/content/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('⚠️  Se observa fuerte desbalance de clases — se aplicará Class Weighting + Augmentation')

In [ ]:
# ──────────────────────────────────────────────
# 2.3 Visualización de imágenes por clase
# ──────────────────────────────────────────────
def load_image(img_id_or_path, directory=None, size=(224, 224)):
    # Accepts full path or id_code + directory
    if directory is not None:
        path = os.path.join(directory, img_id_or_path + '.png')
    else:
        path = img_id_or_path
    img  = cv2.imread(path)
    img  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img  = cv2.resize(img, size)
    return img

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for label_id, (label_name, ax) in enumerate(zip(LABEL_MAP.values(), axes)):
    sample = df_train[df_train['diagnosis'] == label_id].iloc[0]
    img = load_image(sample['image_path'])
    ax.imshow(img)
    ax.set_title(f'Clase {label_id}\n{label_name}', fontsize=10, fontweight='bold')
    ax.axis('off')

plt.suptitle('Ejemplos de Imágenes por Nivel de Severidad', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/content/sample_images_per_class.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ──────────────────────────────────────────────
# 2.4 Preprocesamiento de imágenes médicas
# (Ben Graham preprocessing — técnica estándar para imágenes retinianas)
# ──────────────────────────────────────────────
def preprocess_retinal_image(img_id_or_path, directory=None, size=380):
    """
    Aplica el preprocesamiento Ben Graham:
    - Recorte circular del fondo de ojo
    - Sustracción de Gaussian blur para resaltar estructuras
    - Normalización al rango [0, 1]
    """
    if directory is not None:
        path = os.path.join(directory, img_id_or_path + '.png')
    else:
        path = img_id_or_path
    img  = cv2.imread(path)
    img  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img  = cv2.resize(img, (size, size))

    # Ben Graham: realce de contraste
    img_blur = cv2.GaussianBlur(img, (0, 0), sigmaX=size // 30)
    img_enhanced = cv2.addWeighted(img, 4, img_blur, -4, 128)

    # Máscara circular
    mask = np.zeros(img_enhanced.shape, dtype=np.uint8)
    center = (size // 2, size // 2)
    radius = int(size * 0.45)
    cv2.circle(mask, center, radius, (1, 1, 1), -1)
    img_final = img_enhanced * mask + 128 * (1 - mask)

    return img_final.astype(np.float32) / 255.0

# Comparación visual: Original vs Preprocesada
sample_id = df_train.iloc[0]['image_path']
orig = load_image(sample_id, size=(380, 380))
proc = preprocess_retinal_image(sample_id)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.imshow(orig / 255.0); ax1.set_title('Original', fontweight='bold'); ax1.axis('off')
ax2.imshow(proc);         ax2.set_title('Ben Graham Preprocessed', fontweight='bold'); ax2.axis('off')
plt.suptitle('Preprocesamiento de Imagen Retiniana', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/preprocessing_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Función de preprocesamiento Ben Graham aplicada correctamente')

In [ ]:
# ──────────────────────────────────────────────
# 2.5 Train / Validation Split (Estratificado)
# ──────────────────────────────────────────────
# image_path ya está en df_train desde la carga inicial
# Split estratificado: 80% train / 20% validation
train_df, val_df = train_test_split(
    df_train,
    test_size=0.20,
    random_state=42,
    stratify=df_train['diagnosis']
)

print('=' * 45)
print(f'  Train set : {len(train_df):>5} imágenes')
print(f'  Val   set : {len(val_df):>5} imágenes')
print('=' * 45)
print('\nDistribución de clases en Train:')
print(train_df['diagnosis_label'].value_counts())
print('\n✅ Split estratificado completado')

---
## ⚙️ 3. FEATURE EXTRACTION & ENGINEERING
> **Objetivo:** Construir el pipeline de extracción de características usando Transfer Learning con EfficientNetB4 y Data Augmentation avanzado.

In [ ]:
# ──────────────────────────────────────────────
# 3.1 Data Augmentation (nivel médico)
# ──────────────────────────────────────────────
IMG_SIZE   = 380
BATCH_SIZE = 16
NUM_CLASSES = 5

# Augmentation para entrenamiento (conservador para imágenes médicas)
train_datagen = ImageDataGenerator(
    preprocessing_function=lambda x: x / 255.0,
    rotation_range=360,        # Rotación completa (retina es circular)
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.15,
    width_shift_range=0.10,
    height_shift_range=0.10,
    brightness_range=[0.85, 1.15],
    fill_mode='constant',
    cval=0
)

# Sin augmentation para validación
val_datagen = ImageDataGenerator(
    preprocessing_function=lambda x: x / 255.0
)

# Convertir etiquetas a string para el generador
train_df['diagnosis_str'] = train_df['diagnosis'].astype(str)
val_df['diagnosis_str']   = val_df['diagnosis'].astype(str)

train_generator = train_datagen.flow_from_dataframe(
    dataframe    = train_df,
    x_col        = 'image_path',
    y_col        = 'diagnosis_str',
    target_size  = (IMG_SIZE, IMG_SIZE),
    batch_size   = BATCH_SIZE,
    class_mode   = 'sparse',
    shuffle      = True,
    seed         = 42
)

val_generator = val_datagen.flow_from_dataframe(
    dataframe    = val_df,
    x_col        = 'image_path',
    y_col        = 'diagnosis_str',
    target_size  = (IMG_SIZE, IMG_SIZE),
    batch_size   = BATCH_SIZE,
    class_mode   = 'sparse',
    shuffle      = False
)

print('✅ Generadores de datos configurados')
print(f'   Pasos por epoch (train): {len(train_generator)}')
print(f'   Pasos por epoch (val)  : {len(val_generator)}')

In [ ]:
# ──────────────────────────────────────────────
# 3.2 Visualización del Data Augmentation
# ──────────────────────────────────────────────
aug_gen = ImageDataGenerator(
    rotation_range=360,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.15,
    brightness_range=[0.8, 1.2],
    fill_mode='constant', cval=0
)

sample_img = load_image(train_df.iloc[0]['image_path'], size=(380, 380))
sample_img = sample_img.reshape((1,) + sample_img.shape)

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
axes = axes.flatten()
axes[0].imshow(sample_img[0] / 255.0)
axes[0].set_title('Original', fontweight='bold', color='green')
axes[0].axis('off')

for i, batch in enumerate(aug_gen.flow(sample_img, batch_size=1, seed=i), start=1):
    if i >= 10: break
    axes[i].imshow(batch[0] / 255.0)
    axes[i].set_title(f'Augmented #{i}', fontweight='bold')
    axes[i].axis('off')

plt.suptitle('Data Augmentation — Variaciones generadas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/data_augmentation_samples.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Data Augmentation visualizado')

In [ ]:
# ──────────────────────────────────────────────
# 3.3 Arquitectura: Transfer Learning con EfficientNetB4
# ──────────────────────────────────────────────
def build_efficientnet_model(num_classes=5, img_size=380, learning_rate=1e-4):
    """
    Construye el modelo basado en EfficientNetB4 con capas personalizadas.
    Estrategia: Feature extraction primero, luego fine-tuning.
    """
    # Base preentrenada en ImageNet
    base_model = EfficientNetB4(
        weights     = 'imagenet',
        include_top = False,
        input_shape = (img_size, img_size, 3)
    )
    base_model.trainable = False  # Congelar en fase 1

    # Cabeza de clasificación personalizada
    inputs = keras.Input(shape=(img_size, img_size, 3))
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(512, activation='relu', name='fc1')(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation='relu', name='fc2')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='output')(x)

    model = keras.Model(inputs, outputs, name='EfficientNetB4_RetinalDR')

    model.compile(
        optimizer = keras.optimizers.Adam(learning_rate=learning_rate),
        loss      = 'sparse_categorical_crossentropy',
        metrics   = ['accuracy',
                     keras.metrics.SparseTopKCategoricalAccuracy(k=2, name='top2_acc')]
    )
    return model, base_model

model, base_model = build_efficientnet_model(num_classes=NUM_CLASSES, img_size=IMG_SIZE)
model.summary()
print(f'\n✅ Modelo construido: {model.count_params():,} parámetros totales')
print(f'   Parámetros entrenables: {sum(tf.size(v).numpy() for v in model.trainable_variables):,}')

---
## 📊 4. FEATURE SCALING & SELECTION
> **Objetivo:** Manejar el desbalance de clases, calcular class weights, y configurar callbacks de regularización.

In [ ]:
# ──────────────────────────────────────────────
# 4.1 Manejo de Desbalance de Clases (Class Weighting)
# ──────────────────────────────────────────────
class_weights_array = compute_class_weight(
    class_weight = 'balanced',
    classes      = np.arange(NUM_CLASSES),
    y            = train_df['diagnosis'].values
)
class_weights = {i: w for i, w in enumerate(class_weights_array)}

print('Class Weights calculados (para compensar desbalance):')
print('─' * 40)
for cls_id, weight in class_weights.items():
    print(f'  Clase {cls_id} ({LABEL_MAP[cls_id]:<18}): {weight:.4f}')

# Visualización
fig, ax = plt.subplots(figsize=(8, 4))
classes = [LABEL_MAP[i] for i in range(NUM_CLASSES)]
weights = [class_weights[i] for i in range(NUM_CLASSES)]
bars = ax.bar(classes, weights, color=['#2ecc71','#f39c12','#e67e22','#e74c3c','#8e44ad'])
ax.set_title('Class Weights — Compensación por Desbalance', fontsize=12, fontweight='bold')
ax.set_xlabel('Clase'); ax.set_ylabel('Peso')
for bar, w in zip(bars, weights):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{w:.2f}', ha='center', fontweight='bold')
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('/content/class_weights.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Class Weights configurados')

In [ ]:
# ──────────────────────────────────────────────
# 4.2 Configuración de Callbacks (Regularización del entrenamiento)
# ──────────────────────────────────────────────
import datetime

log_dir = '/content/logs/fit/' + datetime.datetime.now().strftime('%Y%m%d-%H%M%S')

callbacks = [
    # Detiene entrenamiento si val_loss no mejora en 7 epochs
    keras.callbacks.EarlyStopping(
        monitor              = 'val_loss',
        patience             = 7,
        restore_best_weights = True,
        verbose              = 1
    ),
    # Guarda el mejor modelo
    keras.callbacks.ModelCheckpoint(
        filepath         = '/content/best_model_phase1.h5',
        monitor          = 'val_accuracy',
        save_best_only   = True,
        verbose          = 1
    ),
    # Reduce learning rate si val_loss se estanca
    keras.callbacks.ReduceLROnPlateau(
        monitor   = 'val_loss',
        factor    = 0.3,
        patience  = 3,
        min_lr    = 1e-7,
        verbose   = 1
    ),
    # TensorBoard para monitoreo
    keras.callbacks.TensorBoard(
        log_dir          = log_dir,
        histogram_freq   = 1
    )
]

print('✅ Callbacks configurados:')
print('   • EarlyStopping       (patience=7)')
print('   • ModelCheckpoint     (guarda mejor val_accuracy)')
print('   • ReduceLROnPlateau   (factor=0.3, patience=3)')
print('   • TensorBoard         (logs en', log_dir + ')')

In [ ]:
# ──────────────────────────────────────────────
# 4.3 Resumen del Pipeline de Datos — Verificación Final
# ──────────────────────────────────────────────
print('=' * 55)
print('     RESUMEN DEL PIPELINE — ETAPAS COMPLETADAS')
print('=' * 55)
print(f'  Dataset           : APTOS 2019 (Kaggle)')
print(f'  Total imágenes    : {len(df_train)}')
print(f'  Train set         : {len(train_df)} ({len(train_df)/len(df_train)*100:.0f}%)')
print(f'  Val set           : {len(val_df)} ({len(val_df)/len(df_train)*100:.0f}%)')
print(f'  Tamaño imagen     : {IMG_SIZE}x{IMG_SIZE} px')
print(f'  Batch size        : {BATCH_SIZE}')
print(f'  Preprocesamiento  : Ben Graham (médico)')
print(f'  Augmentation      : Rotación 360°, Flip, Zoom, Brillo')
print(f'  Arquitectura      : EfficientNetB4 + Custom Head')
print(f'  Desbalance        : Class Weighting (balanced)')
print(f'  Parámetros modelo : {model.count_params():,}')
print('=' * 55)
print()
print('⏳ PENDIENTE:')
print('  [ ] 5. Modeling — Entrenamiento Fase 1 (feature extraction)')
print('  [ ]    Entrenamiento Fase 2 (fine-tuning capas superiores)')
print('  [ ] 6. Model Evaluation & Tuning')
print('      - Curvas de entrenamiento (loss & accuracy)')
print('      - Matriz de confusión')
print('      - Reporte de clasificación (precision, recall, F1)')
print('      - Curva ROC-AUC multiclase')
print('      - Grad-CAM (interpretabilidad)')
print('      - Hyperparameter tuning')
print('  [ ] 7. Deployment & Monitoring')
print('      - Exportar modelo (.h5 / TFLite)')
print('      - Demo de inferencia con nuevas imágenes')
print('      - Gradio/Streamlit demo (opcional)')

---
## 🤖 5. MODELING — Machine Learning Algorithm
> **Objetivo:** Entrenar EfficientNetB4 en dos fases: primero feature extraction (base congelada), luego fine-tuning de las capas superiores.

In [ ]:
# ──────────────────────────────────────────────
# 5.1 FASE 1 — Feature Extraction (base congelada)
# Solo se entrenan las capas de la cabeza personalizada
# ──────────────────────────────────────────────
print('=' * 55)
print('  FASE 1: Feature Extraction — Base congelada')
print('=' * 55)
print(f'  Capas entrenables: {sum(1 for l in model.layers if l.trainable)}')
print(f'  Learning rate    : 1e-4')
print(f'  Epochs máx       : 20')

history_phase1 = model.fit(
    train_generator,
    validation_data = val_generator,
    epochs          = 20,
    class_weight    = class_weights,
    callbacks       = callbacks,
    verbose         = 1
)
print('\n✅ Fase 1 completada')

In [ ]:
# ──────────────────────────────────────────────
# 5.2 Curvas de entrenamiento — Fase 1
# ──────────────────────────────────────────────
def plot_training_history(history, title='Fase 1 — Feature Extraction'):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Loss
    axes[0].plot(history.history['loss'],     label='Train Loss',      color='#3498db', linewidth=2)
    axes[0].plot(history.history['val_loss'], label='Val Loss',        color='#e74c3c', linewidth=2, linestyle='--')
    axes[0].set_title(f'{title} — Loss', fontweight='bold')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].legend(); axes[0].grid(alpha=0.3)

    # Accuracy
    axes[1].plot(history.history['accuracy'],     label='Train Acc',   color='#2ecc71', linewidth=2)
    axes[1].plot(history.history['val_accuracy'], label='Val Acc',     color='#e67e22', linewidth=2, linestyle='--')
    axes[1].set_title(f'{title} — Accuracy', fontweight='bold')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
    axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.tight_layout()
    fname = title.replace(' ', '_').replace('—', '').replace('/', '') + '.png'
    plt.savefig(f'/content/{fname}', dpi=150, bbox_inches='tight')
    plt.show()

plot_training_history(history_phase1, 'Fase 1 — Feature Extraction')

best_val_acc = max(history_phase1.history['val_accuracy'])
print(f'\n✅ Mejor val_accuracy Fase 1: {best_val_acc:.4f} ({best_val_acc*100:.2f}%)')

In [ ]:
# ──────────────────────────────────────────────
# 5.3 FASE 2 — Fine Tuning (descongelar capas superiores)
# Se descongelan las últimas 50 capas de EfficientNetB4
# con un learning rate muy bajo para no destruir los pesos
# ──────────────────────────────────────────────
print('=' * 55)
print('  FASE 2: Fine Tuning — Capas superiores descongeladas')
print('=' * 55)

# Descongelar base
base_model.trainable = True

# Congelar hasta la capa 300, descongelar desde ahí
fine_tune_at = 300
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

trainable_count = sum(1 for l in model.layers if l.trainable)
print(f'  Capas entrenables: {trainable_count}')
print(f'  Learning rate    : 1e-5 (10x menor que Fase 1)')

# Recompilar con LR más bajo
model.compile(
    optimizer = keras.optimizers.Adam(learning_rate=1e-5),
    loss      = 'sparse_categorical_crossentropy',
    metrics   = ['accuracy',
                 keras.metrics.SparseTopKCategoricalAccuracy(k=2, name='top2_acc')]
)

# Callbacks para fase 2
callbacks_phase2 = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=5,
                                  restore_best_weights=True, verbose=1),
    keras.callbacks.ModelCheckpoint('/content/best_model_phase2.h5',
                                    monitor='val_accuracy', save_best_only=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3,
                                      patience=2, min_lr=1e-8, verbose=1)
]

history_phase2 = model.fit(
    train_generator,
    validation_data = val_generator,
    epochs          = 20,
    class_weight    = class_weights,
    callbacks       = callbacks_phase2,
    verbose         = 1
)
print('\n✅ Fase 2 (Fine Tuning) completada')

In [ ]:
# ──────────────────────────────────────────────
# 5.4 Curvas de entrenamiento — Fase 2 + Comparación
# ──────────────────────────────────────────────
plot_training_history(history_phase2, 'Fase 2 — Fine Tuning')

best_val_acc_ft = max(history_phase2.history['val_accuracy'])
print(f'\n📊 Resumen de mejora:')
print(f'   Val Accuracy Fase 1 (Feature Extraction): {max(history_phase1.history["val_accuracy"]):.4f}')
print(f'   Val Accuracy Fase 2 (Fine Tuning)        : {best_val_acc_ft:.4f}')
print(f'   Mejora                                   : +{(best_val_acc_ft - max(history_phase1.history["val_accuracy"]))*100:.2f}%')

---
## 📈 6. MODEL EVALUATION & TUNING
> **Objetivo:** Evaluar el modelo con métricas clínicas avanzadas, visualizar errores, aplicar Grad-CAM para interpretabilidad y re-iterar si es necesario.

In [ ]:
# ──────────────────────────────────────────────
# 6.1 Predicciones sobre el set de validación
# ──────────────────────────────────────────────
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, roc_curve)
from sklearn.preprocessing import label_binarize
import itertools

# Obtener predicciones
val_generator.reset()
y_pred_proba = model.predict(val_generator, verbose=1)
y_pred       = np.argmax(y_pred_proba, axis=1)
y_true       = val_generator.classes

print(f'\n✅ Predicciones generadas: {len(y_pred)} muestras')
print(f'   Accuracy global: {np.mean(y_pred == y_true):.4f} ({np.mean(y_pred == y_true)*100:.2f}%)')

In [ ]:
# ──────────────────────────────────────────────
# 6.2 Reporte de Clasificación completo
# ──────────────────────────────────────────────
target_names = list(LABEL_MAP.values())
report = classification_report(y_true, y_pred, target_names=target_names, digits=4)
print('CLASSIFICATION REPORT — EfficientNetB4')
print('=' * 65)
print(report)

In [ ]:
# ──────────────────────────────────────────────
# 6.3 Matriz de Confusión
# ──────────────────────────────────────────────
cm = confusion_matrix(y_true, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Matriz absoluta
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_names, yticklabels=target_names, ax=axes[0])
axes[0].set_title('Matriz de Confusión — Valores Absolutos', fontweight='bold')
axes[0].set_ylabel('Real'); axes[0].set_xlabel('Predicho')
axes[0].tick_params(axis='x', rotation=20)

# Matriz normalizada
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Greens',
            xticklabels=target_names, yticklabels=target_names, ax=axes[1])
axes[1].set_title('Matriz de Confusión — Normalizada', fontweight='bold')
axes[1].set_ylabel('Real'); axes[1].set_xlabel('Predicho')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ──────────────────────────────────────────────
# 6.4 Curva ROC-AUC Multiclase (One vs Rest)
# ──────────────────────────────────────────────
y_true_bin = label_binarize(y_true, classes=list(range(NUM_CLASSES)))

fig, ax = plt.subplots(figsize=(9, 7))
colors_roc = ['#3498db', '#2ecc71', '#e67e22', '#e74c3c', '#8e44ad']

auc_scores = []
for i, (color, label) in enumerate(zip(colors_roc, target_names)):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_proba[:, i])
    auc_score   = roc_auc_score(y_true_bin[:, i], y_pred_proba[:, i])
    auc_scores.append(auc_score)
    ax.plot(fpr, tpr, color=color, linewidth=2,
            label=f'{label} (AUC = {auc_score:.4f})')

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('Curva ROC — Multiclase (One vs Rest)', fontsize=13, fontweight='bold')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/content/roc_auc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

macro_auc = np.mean(auc_scores)
print(f'\n📊 Macro-Average AUC-ROC: {macro_auc:.4f}')
for name, score in zip(target_names, auc_scores):
    print(f'   {name:<20}: {score:.4f}')

In [ ]:
# ──────────────────────────────────────────────
# 6.5 Grad-CAM — Interpretabilidad del modelo
# Visualiza qué zonas de la retina activa el modelo
# ──────────────────────────────────────────────
import tensorflow as tf

def make_gradcam_heatmap(img_array, model, last_conv_layer_name='top_conv', pred_index=None):
    """
    Genera mapa de calor Grad-CAM para una imagen dada.
    Resalta las regiones que más influyen en la predicción.
    """
    grad_model = tf.keras.models.Model(
        inputs  = model.inputs,
        outputs = [model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        last_conv_output, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]

    grads    = tape.gradient(class_channel, last_conv_output)
    pooled   = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap  = last_conv_output[0] @ pooled[..., tf.newaxis]
    heatmap  = tf.squeeze(heatmap)
    heatmap  = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

def display_gradcam(img_path, directory, model, true_label, pred_label):
    img_orig = load_image(img_path, size=(IMG_SIZE, IMG_SIZE))
    img_proc = preprocess_retinal_image(img_path, size=IMG_SIZE)
    img_arr  = np.expand_dims(img_proc, axis=0)

    heatmap = make_gradcam_heatmap(img_arr, model)
    heatmap_resized = cv2.resize(heatmap, (IMG_SIZE, IMG_SIZE))
    heatmap_color   = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
    heatmap_color   = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)
    superimposed    = cv2.addWeighted(img_orig, 0.6, heatmap_color, 0.4, 0)

    return img_orig, heatmap_color, superimposed

# Visualizar Grad-CAM para 1 imagen por clase
fig, axes = plt.subplots(NUM_CLASSES, 3, figsize=(12, 4 * NUM_CLASSES))
col_titles = ['Imagen Original', 'Grad-CAM Heatmap', 'Superposición']

for col, title in enumerate(col_titles):
    axes[0, col].set_title(title, fontsize=12, fontweight='bold', pad=10)

for class_id in range(NUM_CLASSES):
    sample    = val_df[val_df['diagnosis'] == class_id].iloc[0]
    img_path  = sample['image_path']
    true_lbl  = LABEL_MAP[class_id]

    img_proc  = preprocess_retinal_image(img_path, size=IMG_SIZE)
    img_arr   = np.expand_dims(img_proc, axis=0)
    pred_prob = model.predict(img_arr, verbose=0)
    pred_lbl  = LABEL_MAP[np.argmax(pred_prob)]

    orig, heatmap, superimposed = display_gradcam(img_path, None, model, true_lbl, pred_lbl)

    axes[class_id, 0].imshow(orig / 255.0)
    axes[class_id, 0].set_ylabel(f'Real: {true_lbl}\nPred: {pred_lbl}',
                                  fontsize=9, fontweight='bold',
                                  color='green' if true_lbl == pred_lbl else 'red')
    axes[class_id, 1].imshow(heatmap)
    axes[class_id, 2].imshow(superimposed / 255.0)
    for col in range(3):
        axes[class_id, col].axis('off')

plt.suptitle('Grad-CAM — Regiones relevantes para la predicción', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('/content/gradcam_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Grad-CAM generado — Las zonas rojas indican las regiones más influyentes para el diagnóstico')

In [ ]:
# ──────────────────────────────────────────────
# 6.6 Re-iteración del Pipeline
# (Re-iterate till satisfactory model performance)
# ──────────────────────────────────────────────
print('=' * 60)
print('  EVALUACIÓN FINAL — DECISIÓN DE RE-ITERACIÓN')
print('=' * 60)

final_acc  = np.mean(y_pred == y_true)
final_auc  = macro_auc
THRESHOLD_ACC = 0.75
THRESHOLD_AUC = 0.90

print(f'\n  Accuracy final    : {final_acc:.4f}  (umbral mínimo: {THRESHOLD_ACC})')
print(f'  Macro AUC-ROC     : {final_auc:.4f}  (umbral mínimo: {THRESHOLD_AUC})')

if final_acc >= THRESHOLD_ACC and final_auc >= THRESHOLD_AUC:
    print('\n  ✅ Rendimiento SATISFACTORIO — Proceder a Deployment')
    PROCEED_TO_DEPLOYMENT = True
else:
    print('\n  ⚠️  Rendimiento INSATISFACTORIO — Se recomienda re-iterar:')
    if final_acc < THRESHOLD_ACC:
        print('     • Aumentar epochs o ajustar learning rate')
        print('     • Revisar Data Augmentation')
    if final_auc < THRESHOLD_AUC:
        print('     • Revisar desbalance de clases')
        print('     • Probar arquitectura más profunda (EfficientNetB5/B6)')
    PROCEED_TO_DEPLOYMENT = False

print('\n  → El pipeline incluye re-iteración automática hacia Data Preparation')
print('    si el rendimiento no alcanza los umbrales definidos.')

---
## 🚀 7. DEPLOYMENT & MONITORING
> **Objetivo:** Exportar el modelo entrenado, crear un pipeline de inferencia para nuevas imágenes y desplegar una demo interactiva.

In [ ]:
# ──────────────────────────────────────────────
# 7.1 Exportar el modelo en múltiples formatos
# ──────────────────────────────────────────────

# Formato .h5 (Keras estándar)
model.save('/content/retinal_dr_efficientnetb4.h5')
print('✅ Modelo guardado: retinal_dr_efficientnetb4.h5')

# Formato SavedModel (TensorFlow)
model.save('/content/retinal_dr_savedmodel')
print('✅ Modelo guardado: retinal_dr_savedmodel/')

# Formato TFLite (para dispositivos móviles / edge)
converter    = tf.lite.TFLiteConverter.from_saved_model('/content/retinal_dr_savedmodel')
converter.optimizations = [tf.lite.Optimize.DEFAULT]  # Quantización dinámica
tflite_model = converter.convert()

with open('/content/retinal_dr_model.tflite', 'wb') as f:
    f.write(tflite_model)
print('✅ Modelo TFLite guardado: retinal_dr_model.tflite')

# Tamaños
import os
h5_size     = os.path.getsize('/content/retinal_dr_efficientnetb4.h5') / (1024**2)
tflite_size = os.path.getsize('/content/retinal_dr_model.tflite') / (1024**2)
print(f'\n  Tamaño .h5     : {h5_size:.1f} MB')
print(f'  Tamaño .tflite : {tflite_size:.1f} MB  (comprimido para deployment)')

In [ ]:
# ──────────────────────────────────────────────
# 7.2 Pipeline de inferencia para nuevas imágenes
# ──────────────────────────────────────────────
def predict_retinal_image(image_path, model, img_size=380, threshold=0.5):
    """
    Pipeline completo de inferencia para una nueva imagen retiniana.
    Retorna diagnóstico, nivel de confianza y alerta clínica.
    """
    # 1. Cargar imagen
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (img_size, img_size))

    # 2. Preprocesamiento Ben Graham
    img_blur     = cv2.GaussianBlur(img, (0, 0), sigmaX=img_size // 30)
    img_enhanced = cv2.addWeighted(img, 4, img_blur, -4, 128)
    mask         = np.zeros(img_enhanced.shape, dtype=np.uint8)
    cv2.circle(mask, (img_size // 2, img_size // 2), int(img_size * 0.45), (1, 1, 1), -1)
    img_proc     = (img_enhanced * mask + 128 * (1 - mask)).astype(np.float32) / 255.0

    # 3. Predicción
    img_arr   = np.expand_dims(img_proc, axis=0)
    probs     = model.predict(img_arr, verbose=0)[0]
    pred_cls  = np.argmax(probs)
    confidence = probs[pred_cls]

    # 4. Alerta clínica
    clinical_alert = {
        0: '🟢 Sin acción inmediata requerida',
        1: '🟡 Seguimiento en 12 meses',
        2: '🟠 Referir a oftalmólogo en 6 meses',
        3: '🔴 Referir urgente a oftalmólogo',
        4: '🚨 URGENCIA — Riesgo de pérdida de visión'
    }

    result = {
        'diagnosis'     : LABEL_MAP[pred_cls],
        'severity_level': pred_cls,
        'confidence'    : float(confidence),
        'clinical_alert': clinical_alert[pred_cls],
        'probabilities' : {LABEL_MAP[i]: float(p) for i, p in enumerate(probs)}
    }
    return result

# Demo con una imagen del set de validación
sample_path = val_df.iloc[0]['image_path']
true_label  = LABEL_MAP[val_df.iloc[0]['diagnosis']]
result      = predict_retinal_image(sample_path, model)

print('=' * 50)
print('  DEMO — INFERENCIA EN NUEVA IMAGEN')
print('=' * 50)
print(f'  Diagnóstico real   : {true_label}')
print(f'  Diagnóstico modelo : {result["diagnosis"]}')
print(f'  Nivel de severidad : {result["severity_level"]}')
print(f'  Confianza          : {result["confidence"]*100:.2f}%')
print(f'  Alerta clínica     : {result["clinical_alert"]}')
print('\n  Distribución de probabilidades:')
for cls_name, prob in result['probabilities'].items():
    bar = '█' * int(prob * 30)
    print(f'    {cls_name:<20}: {bar:<30} {prob*100:.1f}%')

In [ ]:
# ──────────────────────────────────────────────
# 7.3 Demo Interactivo con Gradio
# ──────────────────────────────────────────────
!pip install -q gradio
import gradio as gr

def gradio_predict(image):
    """Función de predicción para la interfaz Gradio."""
    img = cv2.resize(image, (IMG_SIZE, IMG_SIZE)).astype(np.float32)
    img_blur     = cv2.GaussianBlur(img.astype(np.uint8), (0, 0), sigmaX=IMG_SIZE // 30)
    img_enhanced = cv2.addWeighted(img.astype(np.uint8), 4, img_blur, -4, 128)
    mask         = np.zeros(img_enhanced.shape, dtype=np.uint8)
    cv2.circle(mask, (IMG_SIZE // 2, IMG_SIZE // 2), int(IMG_SIZE * 0.45), (1, 1, 1), -1)
    img_proc     = (img_enhanced * mask + 128 * (1 - mask)).astype(np.float32) / 255.0

    probs    = model.predict(np.expand_dims(img_proc, 0), verbose=0)[0]
    pred_cls = np.argmax(probs)

    alert_map = {
        0: '🟢 Sin acción inmediata',
        1: '🟡 Seguimiento en 12 meses',
        2: '🟠 Referir en 6 meses',
        3: '🔴 Referir urgente',
        4: '🚨 URGENCIA médica'
    }

    output_text = (
        f"Diagnóstico: {LABEL_MAP[pred_cls]}\n"
        f"Severidad  : Nivel {pred_cls}/4\n"
        f"Confianza  : {probs[pred_cls]*100:.1f}%\n"
        f"Alerta     : {alert_map[pred_cls]}"
    )
    prob_dict = {LABEL_MAP[i]: float(p) for i, p in enumerate(probs)}
    return output_text, prob_dict

demo = gr.Interface(
    fn          = gradio_predict,
    inputs      = gr.Image(label='Imagen de Fondo de Ojo', type='numpy'),
    outputs     = [
        gr.Textbox(label='Diagnóstico'),
        gr.Label(label='Probabilidades por Clase', num_top_classes=5)
    ],
    title       = '🧠 Detección de Retinopatía Diabética',
    description = 'Sube una imagen del fondo de ojo para obtener el diagnóstico automático usando EfficientNetB4.',
    theme       = gr.themes.Soft()
)

demo.launch(share=True, debug=False)
print('\n✅ Demo Gradio activa — Usa el link público generado arriba')

In [ ]:
# ──────────────────────────────────────────────
# 7.4 Resumen Final del Proyecto
# ──────────────────────────────────────────────
print('=' * 60)
print('   RESUMEN FINAL — PIPELINE COMPLETO')
print('=' * 60)
print()
print('  ✅ 1. Data Retrieval')
print('        APTOS 2019 — 3,662 imágenes retinianas (Kaggle)')
print()
print('  ✅ 2. Data Processing & Wrangling')
print('        EDA, preprocesamiento Ben Graham, split 80/20')
print()
print('  ✅ 3. Feature Extraction & Engineering')
print('        Data Augmentation médico, Transfer Learning EfficientNetB4')
print()
print('  ✅ 4. Feature Scaling & Selection')
print('        Class Weighting, Callbacks (EarlyStopping, ReduceLR)')
print()
print('  ✅ 5. Modeling')
print('        Fase 1: Feature Extraction | Fase 2: Fine Tuning')
print(f'        Val Accuracy: {final_acc:.4f} | AUC-ROC: {final_auc:.4f}')
print()
print('  ✅ 6. Model Evaluation & Tuning')
print('        Confusion Matrix, ROC-AUC multiclase, Grad-CAM')
print()
print('  ✅ 7. Deployment & Monitoring')
print('        Exportado: .h5, SavedModel, TFLite')
print('        Demo interactivo: Gradio')
print()
print('=' * 60)
print('  🎓 Proyecto de Maestría — Deep Learning')
print('     Detección de Retinopatía Diabética con EfficientNetB4')
print('=' * 60)